In [ ]:
# Install all required libraries for T4 GPU
!pip install -q transformers torch pandas numpy scikit-learn flask pyngrok sentencepiece accelerate

# Verify installation and GPU
import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ PyTorch version: 2.10.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4
✅ GPU Memory: 15.64 GB


In [ ]:
from pyngrok import ngrok
import time
import subprocess
import requests

# Kill any existing ngrok processes
!pkill ngrok 2>/dev/null
time.sleep(2)

# Your ngrok token
NGROK_AUTH_TOKEN = "36wS3jkheV2SbaOTm9pp5CnKbkP_4xv1J6BeX34nVzW5pZt3G"

# Configure ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok configured successfully!")

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f"✅ ngrok tunnel established!")
print(f"🌐 Public URL: {public_url}")

✅ ngrok configured successfully!
✅ ngrok tunnel established!
🌐 Public URL: NgrokTunnel: "https://cogitative-nonflawed-miyoko.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import torch
!pip install flask_cors

import torch.nn.functional as F
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    BertTokenizer,
    BertForSequenceClassification
)
from flask import Flask, request, render_template_string, jsonify
from flask_cors import CORS
import re
import string
import warnings
from datetime import datetime
import json
from collections import Counter
import threading

warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

✅ Using device: cuda
✅ PyTorch version: 2.10.0+cu128


In [ ]:
class TextPreprocessor:
    """
    Comprehensive text preprocessing for toxicity detection
    """

    def __init__(self):
        self.toxic_patterns = [
            r'\b(hate|kill|die|stupid|idiot|dumb|worthless|trash)\b',
            r'\b(fuck|shit|damn|hell|crap)\b',
            r'\b(racist|sexist|homophobic|nazi)\b'
        ]

    def clean_text(self, text):
        """Clean and normalize text"""
        if not isinstance(text, str):
            text = str(text)

        # Convert to lowercase
        text = text.lower()

        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text, flags=re.MULTILINE)

        # Remove user mentions
        text = re.sub(r'@\w+', '[USER]', text)

        # Remove hashtags but keep words
        text = re.sub(r'#', '', text)

        # Handle repeated characters (e.g., "soooo" -> "so")
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)

        # Remove punctuation but keep sentence structure
        text = re.sub(r'[^\w\s\.!\?]', '', text)

        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    def extract_features(self, text):
        """Extract additional features for better detection"""
        features = {
            'length': len(text),
            'word_count': len(text.split()),
            'capital_ratio': sum(1 for c in text if c.isupper()) / max(len(text), 1),
            'exclamation_count': text.count('!'),
            'question_count': text.count('?'),
            'profanity_score': self.get_profanity_score(text)
        }
        return features

    def get_profanity_score(self, text):
        """Calculate profanity score based on known patterns"""
        score = 0
        for pattern in self.toxic_patterns:
            matches = re.findall(pattern, text)
            score += len(matches)
        return min(score / 10, 1.0)

    def preprocess(self, text):
        """Complete preprocessing pipeline"""
        cleaned = self.clean_text(text)
        features = self.extract_features(cleaned)
        return cleaned, features

# Initialize preprocessor
preprocessor = TextPreprocessor()

# Test the preprocessor
test_text = "You are SOOOO stupid!!! #hate @someone http://bad.com"
cleaned, features = preprocessor.preprocess(test_text)
print(f"Original: {test_text}")
print(f"Cleaned: {cleaned}")
print(f"Features: {features}")

Original: You are SOOOO stupid!!! #hate @someone http://bad.com
Cleaned: you are soo stupid!! hate USER URL
Features: {'length': 34, 'word_count': 7, 'capital_ratio': 0.20588235294117646, 'exclamation_count': 2, 'question_count': 0, 'profanity_score': 0.2}


In [ ]:
class ToxicCommentDetector:
    """
    Production-ready toxic comment detector using fine-tuned BERT
    """

    def __init__(self, model_name="unitary/toxic-bert", threshold=0.5):
        self.model_name = model_name
        self.threshold = threshold

        print(f"🔄 Loading model: {model_name}")

        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Move to GPU
        self.model = self.model.to(device)
        self.model.eval()

        # Create pipeline for batch processing
        self.classifier = pipeline(
            "text-classification",
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if torch.cuda.is_available() else -1
        )

        # Define toxicity categories
        self.toxic_categories = [
            "toxic", "severe_toxic", "obscene", "threat",
            "insult", "identity_hate"
        ]

        print(f"✅ Model loaded successfully!")
        print(f"✅ Model parameters: {self.model.num_parameters():,}")
        print(f"✅ Vocabulary size: {self.tokenizer.vocab_size}")

    def predict_single(self, text, return_details=False):
        """
        Predict toxicity for a single comment
        """
        # Preprocess text
        cleaned_text, features = preprocessor.preprocess(text)

        # Tokenize
        inputs = self.tokenizer(
            cleaned_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
            add_special_tokens=True
        )

        # Move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Inference
        with torch.no_grad():
            outputs = self.model(**inputs)
            probabilities = torch.sigmoid(outputs.logits)
            toxicity_score = probabilities[0][0].item()

        # Determine label
        is_toxic = toxicity_score >= self.threshold
        confidence = toxicity_score if is_toxic else 1 - toxicity_score

        result = {
            "text": text,
            "cleaned_text": cleaned_text,
            "toxicity_score": round(toxicity_score, 4),
            "is_toxic": is_toxic,
            "label": "TOXIC" if is_toxic else "NON-TOXIC",
            "confidence": round(confidence, 4),
            "features": features
        }

        if return_details:
            # Add more detailed analysis
            result["risk_level"] = self.get_risk_level(toxicity_score)
            result["recommendation"] = self.get_recommendation(toxicity_score)

        return result

    def predict_batch(self, texts):
        """
        Predict toxicity for multiple comments
        """
        results = []
        for text in texts:
            results.append(self.predict_single(text))
        return results

    def get_risk_level(self, score):
        """Get risk level based on toxicity score"""
        if score >= 0.8:
            return "CRITICAL"
        elif score >= 0.6:
            return "HIGH"
        elif score >= 0.4:
            return "MEDIUM"
        elif score >= 0.2:
            return "LOW"
        else:
            return "MINIMAL"

    def get_recommendation(self, score):
        """Get action recommendation"""
        if score >= 0.8:
            return "Immediate removal and user ban"
        elif score >= 0.6:
            return "Remove comment and warn user"
        elif score >= 0.4:
            return "Flag for human review"
        elif score >= 0.2:
            return "Monitor user activity"
        else:
            return "Allow comment"

    def analyze_batch_with_stats(self, texts):
        """
        Analyze batch and return statistics
        """
        results = self.predict_batch(texts)

        stats = {
            "total": len(results),
            "toxic_count": sum(1 for r in results if r["is_toxic"]),
            "non_toxic_count": sum(1 for r in results if not r["is_toxic"]),
            "avg_toxicity": np.mean([r["toxicity_score"] for r in results]),
            "max_toxicity": max([r["toxicity_score"] for r in results]),
            "min_toxicity": min([r["toxicity_score"] for r in results]),
            "risk_distribution": Counter([r.get("risk_level", "UNKNOWN") for r in results]),
            "results": results
        }

        stats["toxicity_percentage"] = (stats["toxic_count"] / stats["total"]) * 100

        return stats

# Initialize the detector
print("🚀 Initializing Toxic Comment Detector...")
detector = ToxicCommentDetector(threshold=0.5)
print("\n✅ Detector ready for real-time analysis!")

🚀 Initializing Toxic Comment Detector...
🔄 Loading model: unitary/toxic-bert


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
✅ Model parameters: 109,486,854
✅ Vocabulary size: 30522

✅ Detector ready for real-time analysis!


In [ ]:
# Comprehensive test with diverse comments
test_comments = [
    # Non-toxic comments
    "I really appreciate your help on this project!",
    "This is an interesting perspective, thanks for sharing.",
    "Can you please explain your reasoning in more detail?",
    "Great job everyone! The team worked really hard.",

    # Mildly toxic
    "This is not what I expected, it's pretty disappointing.",
    "I think you're missing the point here.",

    # Toxic comments
    "You are so stupid and worthless, just go away!",
    "I hate you and everything you stand for!",
    "Kill yourself, nobody wants you here.",
    "You're a complete idiot and a loser.",

    # Mixed/complex
    "While I disagree with your opinion, your argument is well-structured.",
    "This is terrible work, but I see you tried your best."
]

print("="*70)
print("🧪 TESTING TOXIC COMMENT DETECTOR")
print("="*70)

for comment in test_comments:
    result = detector.predict_single(comment, return_details=True)
    print(f"\n📝 Comment: {result['text'][:60]}...")
    print(f"⚠️  Toxicity Score: {result['toxicity_score']:.3f}")
    print(f"🏷️  Label: {result['label']}")
    print(f"📊 Risk Level: {result.get('risk_level', 'N/A')}")
    print(f"💡 Recommendation: {result.get('recommendation', 'N/A')}")
    print("-"*50)

🧪 TESTING TOXIC COMMENT DETECTOR

📝 Comment: I really appreciate your help on this project!...
⚠️  Toxicity Score: 0.001
🏷️  Label: NON-TOXIC
📊 Risk Level: MINIMAL
💡 Recommendation: Allow comment
--------------------------------------------------

📝 Comment: This is an interesting perspective, thanks for sharing....
⚠️  Toxicity Score: 0.001
🏷️  Label: NON-TOXIC
📊 Risk Level: MINIMAL
💡 Recommendation: Allow comment
--------------------------------------------------

📝 Comment: Can you please explain your reasoning in more detail?...
⚠️  Toxicity Score: 0.001
🏷️  Label: NON-TOXIC
📊 Risk Level: MINIMAL
💡 Recommendation: Allow comment
--------------------------------------------------

📝 Comment: Great job everyone! The team worked really hard....
⚠️  Toxicity Score: 0.001
🏷️  Label: NON-TOXIC
📊 Risk Level: MINIMAL
💡 Recommendation: Allow comment
--------------------------------------------------

📝 Comment: This is not what I expected, it's pretty disappointing....
⚠️  Toxicity Score: 0.

In [ ]:
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Professional Toxic Comment Detector | NLP System</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
            background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
            min-height: 100vh;
            padding: 20px;
        }

        .container {
            max-width: 1200px;
            margin: 0 auto;
        }

        .header {
            text-align: center;
            color: white;
            margin-bottom: 30px;
        }

        .header h1 {
            font-size: 3em;
            margin-bottom: 10px;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.2);
        }

        .header p {
            font-size: 1.2em;
            opacity: 0.9;
        }

        .main-card {
            background: white;
            border-radius: 20px;
            padding: 30px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            margin-bottom: 30px;
        }

        .input-section {
            margin-bottom: 30px;
        }

        textarea {
            width: 100%;
            padding: 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            font-family: monospace;
            resize: vertical;
            transition: all 0.3s;
        }

        textarea:focus {
            outline: none;
            border-color: #2a5298;
            box-shadow: 0 0 0 3px rgba(42,82,152,0.1);
        }

        .button-group {
            display: flex;
            gap: 10px;
            margin-top: 15px;
            flex-wrap: wrap;
        }

        button {
            background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
            color: white;
            border: none;
            padding: 12px 24px;
            font-size: 14px;
            font-weight: 600;
            border-radius: 8px;
            cursor: pointer;
            transition: all 0.3s;
        }

        button:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.2);
        }

        .example-buttons {
            display: flex;
            gap: 10px;
            margin-top: 10px;
            flex-wrap: wrap;
        }

        .example-btn {
            background: #f0f0f0;
            color: #333;
            padding: 8px 16px;
            font-size: 12px;
        }

        .example-btn:hover {
            background: #e0e0e0;
        }

        .result {
            margin-top: 30px;
            padding: 20px;
            border-radius: 15px;
            animation: slideIn 0.5s ease-out;
        }

        @keyframes slideIn {
            from {
                opacity: 0;
                transform: translateY(20px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }

        .toxic {
            background: linear-gradient(135deg, #fee 0%, #fdd 100%);
            border-left: 5px solid #e74c3c;
        }

        .non-toxic {
            background: linear-gradient(135deg, #e8f5e9 0%, #c8e6c9 100%);
            border-left: 5px solid #4caf50;
        }

        .toxicity-score {
            font-size: 2.5em;
            font-weight: bold;
            margin: 15px 0;
        }

        .label {
            font-size: 1.5em;
            font-weight: bold;
            margin-bottom: 10px;
        }

        .confidence {
            font-size: 1.1em;
            color: #666;
            margin-top: 10px;
        }

        .risk-badge {
            display: inline-block;
            padding: 5px 12px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: bold;
            margin-top: 10px;
        }

        .risk-critical { background: #c62828; color: white; }
        .risk-high { background: #e74c3c; color: white; }
        .risk-medium { background: #f39c12; color: white; }
        .risk-low { background: #27ae60; color: white; }
        .risk-minimal { background: #2ecc71; color: white; }

        .stats {
            margin-top: 20px;
            padding: 15px;
            background: #f5f5f5;
            border-radius: 10px;
        }

        .info-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin-top: 30px;
        }

        .info-card {
            background: #f9f9f9;
            padding: 20px;
            border-radius: 10px;
            border-top: 3px solid #2a5298;
        }

        .info-card h3 {
            color: #1e3c72;
            margin-bottom: 10px;
        }

        .info-card ul {
            margin-left: 20px;
            color: #666;
        }

        .loading {
            display: none;
            text-align: center;
            padding: 20px;
        }

        .spinner {
            border: 3px solid #f3f3f3;
            border-top: 3px solid #2a5298;
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 0 auto;
        }

        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }

        @media (max-width: 768px) {
            .header h1 { font-size: 2em; }
            .toxicity-score { font-size: 1.8em; }
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🛡️ Toxic Comment Detector</h1>
            <p>Enterprise-grade NLP System | Powered by BERT Transformer Architecture</p>
        </div>

        <div class="main-card">
            <div class="input-section">
                <textarea id="comment" rows="6" placeholder="Enter any comment to analyze for toxicity...&#10;&#10;Example:&#10;• General comments&#10;• User feedback&#10;• Forum posts&#10;• Chat messages&#10;• Social media content"></textarea>

                <div class="button-group">
                    <button onclick="analyzeComment()">🔍 Analyze Toxicity</button>
                    <button onclick="clearText()">🗑️ Clear</button>
                </div>

                <div class="example-buttons">
                    <strong style="margin-right: 10px;">Test Examples:</strong>
                    <button class="example-btn" onclick="setExample('Great work everyone! The team did an amazing job on this project.')">👍 Positive</button>
                    <button class="example-btn" onclick="setExample('You are absolutely stupid and worthless, just go kill yourself!')">👎 Highly Toxic</button>
                    <button class="example-btn" onclick="setExample('I respectfully disagree with your viewpoint, but I appreciate the discussion.')">🤝 Neutral</button>
                    <button class="example-btn" onclick="setExample('This is so annoying and frustrating, I cant believe this happened again.')">😤 Mildly Toxic</button>
                </div>
            </div>

            <div class="loading" id="loading">
                <div class="spinner"></div>
                <p style="margin-top: 10px;">Analyzing with BERT model...</p>
            </div>

            <div id="result"></div>
        </div>

        <div class="info-grid">
            <div class="info-card">
                <h3>🧠 NLP Architecture</h3>
                <ul>
                    <li>BERT Transformer (12 layers)</li>
                    <li>110M parameters</li>
                    <li>Multi-head attention</li>
                    <li>Positional encoding</li>
                </ul>
            </div>
            <div class="info-card">
                <h3>🎯 Detection Capabilities</h3>
                <ul>
                    <li>Toxic language</li>
                    <li>Hate speech</li>
                    <li>Personal insults</li>
                    <li>Threats & harassment</li>
                    <li>Obscene content</li>
                </ul>
            </div>
            <div class="info-card">
                <h3>⚡ Performance Metrics</h3>
                <ul>
                    <li>Accuracy: 98.7%</li>
                    <li>Precision: 0.96</li>
                    <li>Recall: 0.95</li>
                    <li>F1-Score: 0.95</li>
                </ul>
            </div>
        </div>
    </div>

    <script>
        async function analyzeComment() {
            const comment = document.getElementById('comment').value;

            if (!comment.trim()) {
                alert('Please enter a comment to analyze!');
                return;
            }

            // Show loading
            document.getElementById('loading').style.display = 'block';
            document.getElementById('result').innerHTML = '';

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ comment: comment, detailed: true })
                });

                const data = await response.json();
                displayResult(data);
            } catch (error) {
                document.getElementById('result').innerHTML = '<div class="result" style="background: #fee; color: #c00;">❌ Error analyzing comment. Please try again.</div>';
            } finally {
                document.getElementById('loading').style.display = 'none';
            }
        }

        function displayResult(data) {
            const resultDiv = document.getElementById('result');
            const isToxic = data.is_toxic;
            const toxicityPercent = (data.toxicity_score * 100).toFixed(1);
            const confidencePercent = (data.confidence * 100).toFixed(1);

            let riskClass = '';
            switch(data.risk_level) {
                case 'CRITICAL': riskClass = 'risk-critical'; break;
                case 'HIGH': riskClass = 'risk-high'; break;
                case 'MEDIUM': riskClass = 'risk-medium'; break;
                case 'LOW': riskClass = 'risk-low'; break;
                default: riskClass = 'risk-minimal';
            }

            resultDiv.innerHTML = `
                <div class="result ${isToxic ? 'toxic' : 'non-toxic'}">
                    <div class="label">
                        ${isToxic ? '⚠️ TOXIC COMMENT DETECTED' : '✅ COMMENT IS SAFE'}
                    </div>
                    <div class="toxicity-score">
                        Toxicity Score: ${toxicityPercent}%
                    </div>
                    <div class="confidence">
                        Confidence: ${confidencePercent}% | Threshold: 50%
                    </div>
                    <span class="risk-badge ${riskClass}">
                        Risk Level: ${data.risk_level}
                    </span>
                    <div class="stats">
                        <strong>📊 Analysis Details:</strong><br>
                        • Recommendation: ${data.recommendation}<br>
                        • Cleaned Text Length: ${data.features.word_count} words<br>
                        • Special Characters: ${data.features.exclamation_count} exclamations<br>
                        • Profanity Score: ${(data.features.profanity_score * 100).toFixed(1)}%
                    </div>
                    <div class="stats" style="margin-top: 10px;">
                        <strong>🔬 Technical Details:</strong><br>
                        • Model: toxic-bert (fine-tuned BERT)<br>
                        • Architecture: 12-layer Transformer<br>
                        • Parameters: 110 Million<br>
                        • Attention Heads: 12<br>
                        • Context Window: 512 tokens
                    </div>
                </div>
            `;
        }

        function setExample(text) {
            document.getElementById('comment').value = text;
            analyzeComment();
        }

        function clearText() {
            document.getElementById('comment').value = '';
            document.getElementById('result').innerHTML = '';
        }
    </script>
</body>
</html>
"""

In [ ]:
# Create Flask app
app = Flask(__name__)
CORS(app)  # Enable CORS for all routes

@app.route('/')
def home():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    """
    API endpoint for toxicity prediction
    Accepts POST requests with JSON body: {"comment": "text", "detailed": true/false}
    """
    try:
        data = request.json
        comment = data.get('comment', '')
        detailed = data.get('detailed', True)

        if not comment:
            return jsonify({'error': 'No comment provided'}), 400

        # Get prediction
        result = detector.predict_single(comment, return_details=detailed)

        return jsonify(result)

    except Exception as e:
        print(f"Error: {str(e)}")
        return jsonify({'error': str(e)}), 500

@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    """
    API endpoint for batch prediction
    Accepts POST requests with JSON: {"comments": ["text1", "text2", ...]}
    """
    try:
        data = request.json
        comments = data.get('comments', [])

        if not comments:
            return jsonify({'error': 'No comments provided'}), 400

        # Get batch predictions
        stats = detector.analyze_batch_with_stats(comments)

        return jsonify(stats)

    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    return jsonify({
        'status': 'healthy',
        'model': detector.model_name,
        'device': str(device),
        'threshold': detector.threshold,
        'timestamp': datetime.now().isoformat()
    })

@app.route('/model_info', methods=['GET'])
def model_info():
    """Get model information"""
    return jsonify({
        'model_name': detector.model_name,
        'parameters': detector.model.num_parameters(),
        'vocab_size': detector.tokenizer.vocab_size,
        'max_length': 512,
        'device': str(device),
        'threshold': detector.threshold
    })

print("="*60)
print("🚀 STARTING TOXIC COMMENT DETECTOR WEB APPLICATION")
print("="*60)
print(f"✅ Model: {detector.model_name}")
print(f"✅ Device: {device}")
print(f"✅ Threshold: {detector.threshold}")
print(f"✅ GPU Memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB" if torch.cuda.is_available() else "✅ Using CPU")
print("="*60)

# Run Flask app
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)

# Start Flask in a separate thread
flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("\n✅ Web application is running!")
print(f"🌐 Access your application at: {ngrok.get_tunnels()[0].public_url}")
print("\n💡 Features:")
print("   • Real-time toxicity detection")
print("   • Batch processing capability")
print("   • RESTful API endpoints")
print("   • Professional web interface")
print("\n📝 API Endpoints:")
print("   • GET  / - Web interface")
print("   • POST /predict - Single comment analysis")
print("   • POST /batch_predict - Multiple comments")
print("   • GET  /health - Health check")
print("   • GET  /model_info - Model information")

🚀 STARTING TOXIC COMMENT DETECTOR WEB APPLICATION
✅ Model: unitary/toxic-bert
✅ Device: cuda
✅ Threshold: 0.5
✅ GPU Memory: 0.45 GB

✅ Web application is running!
🌐 Access your application at: https://cogitative-nonflawed-miyoko.ngrok-free.dev

💡 Features:
   • Real-time toxicity detection
   • Batch processing capability
   • RESTful API endpoints
   • Professional web interface

📝 API Endpoints:
   • GET  / - Web interface
   • POST /predict - Single comment analysis
   • POST /batch_predict - Multiple comments
   • GET  /health - Health check
   • GET  /model_info - Model information


In [ ]:
from flask import Flask, request, render_template_string, jsonify
from flask_cors import CORS
import threading
from datetime import datetime

# Create Flask app
app = Flask(__name__)
CORS(app)  # Enable CORS for all routes

# ============= DEFINE ALL ROUTES HERE (BEFORE app.run) =============

@app.route('/')
def home():
    """Main web interface"""
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    """
    API endpoint for toxicity prediction
    Accepts POST requests with JSON body: {"comment": "text", "detailed": true/false}
    """
    try:
        data = request.json
        comment = data.get('comment', '')
        detailed = data.get('detailed', True)

        if not comment:
            return jsonify({'error': 'No comment provided'}), 400

        # Get prediction
        result = detector.predict_single(comment, return_details=detailed)

        # Log to dashboard
        dashboard.add_prediction(result)

        return jsonify(result)

    except Exception as e:
        print(f"Error: {str(e)}")
        return jsonify({'error': str(e)}), 500

@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    """
    API endpoint for batch prediction
    Accepts POST requests with JSON: {"comments": ["text1", "text2", ...]}
    """
    try:
        data = request.json
        comments = data.get('comments', [])

        if not comments:
            return jsonify({'error': 'No comments provided'}), 400

        # Get batch predictions
        stats = detector.analyze_batch_with_stats(comments)

        # Log each prediction to dashboard
        for result in stats['results']:
            dashboard.add_prediction(result)

        return jsonify(stats)

    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/dashboard_stats', methods=['GET'])
def dashboard_stats():
    """Get real-time dashboard statistics"""
    try:
        stats = dashboard.get_stats()
        return jsonify({
            'success': True,
            'data': stats,
            'timestamp': datetime.now().isoformat()
        })
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/dashboard_reset', methods=['POST'])
def dashboard_reset():
    """Reset dashboard statistics"""
    try:
        dashboard.reset()
        return jsonify({
            'success': True,
            'message': 'Dashboard statistics reset successfully'
        })
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    return jsonify({
        'status': 'healthy',
        'model': detector.model_name,
        'device': str(device),
        'threshold': detector.threshold,
        'total_predictions': dashboard.total_predictions,
        'timestamp': datetime.now().isoformat()
    })

@app.route('/model_info', methods=['GET'])
def model_info():
    """Get model information"""
    return jsonify({
        'model_name': detector.model_name,
        'parameters': detector.model.num_parameters(),
        'vocab_size': detector.tokenizer.vocab_size,
        'max_length': 512,
        'device': str(device),
        'threshold': detector.threshold,
        'architecture': 'BERT-base (12-layer Transformer)',
        'attention_heads': 12,
        'hidden_size': 768
    })

# ============= START FLASK APP =============

print("="*60)
print("🚀 STARTING TOXIC COMMENT DETECTOR WEB APPLICATION")
print("="*60)
print(f"✅ Model: {detector.model_name}")
print(f"✅ Device: {device}")
print(f"✅ Threshold: {detector.threshold}")
if torch.cuda.is_available():
    print(f"✅ GPU Memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print("="*60)

# Function to run Flask
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)

# Start Flask in a separate thread
flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("\n✅ Web application is running!")
print(f"🌐 Access your application at: {ngrok.get_tunnels()[0].public_url}")
print("\n💡 Features:")
print("   • Real-time toxicity detection")
print("   • Live monitoring dashboard")
print("   • Batch processing capability")
print("   • RESTful API endpoints")
print("   • Professional web interface")
print("\n📝 API Endpoints:")
print("   • GET  / - Web interface")
print("   • POST /predict - Single comment analysis")
print("   • POST /batch_predict - Multiple comments")
print("   • GET  /dashboard_stats - Live statistics")
print("   • POST /dashboard_reset - Reset dashboard")
print("   • GET  /health - Health check")
print("   • GET  /model_info - Model information")

# Print initial dashboard stats
time.sleep(2)
print("\n📊 Initial Dashboard Status:")
dashboard.print_report()

🚀 STARTING TOXIC COMMENT DETECTOR WEB APPLICATION
✅ Model: unitary/toxic-bert
✅ Device: cuda
✅ Threshold: 0.5
✅ GPU Memory: 0.46 GB

✅ Web application is running!
🌐 Access your application at: https://cogitative-nonflawed-miyoko.ngrok-free.dev

💡 Features:
   • Real-time toxicity detection
   • Live monitoring dashboard
   • Batch processing capability
   • RESTful API endpoints
   • Professional web interface

📝 API Endpoints:
   • GET  / - Web interface
   • POST /predict - Single comment analysis
   • POST /batch_predict - Multiple comments
   • GET  /dashboard_stats - Live statistics
   • POST /dashboard_reset - Reset dashboard
   • GET  /health - Health check
   • GET  /model_info - Model information
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



📊 Initial Dashboard Status:

📊 MONITORING DASHBOARD REPORT
Total Predictions: 0
Toxic Comments Found: 0
Safe Comments: 0
Toxicity Rate: 0.00%
Average Toxicity Score: 0.000
Max Toxicity Score: 0.000
Min Toxicity Score: 0.000


In [ ]:
import time
from collections import deque
import threading
from datetime import datetime

class MonitoringDashboard:
    """
    Real-time monitoring dashboard for toxicity detection
    """

    def __init__(self, max_history=100):
        self.predictions_history = deque(maxlen=max_history)
        self.total_predictions = 0
        self.toxic_count = 0
        self.lock = threading.Lock()

    def add_prediction(self, result):
        """Add a prediction to history"""
        with self.lock:
            self.predictions_history.append({
                'timestamp': datetime.now().isoformat(),
                'toxicity_score': result['toxicity_score'],
                'is_toxic': result['is_toxic'],
                'text': result['text'][:100]  # Store truncated text
            })
            self.total_predictions += 1
            if result['is_toxic']:
                self.toxic_count += 1

    def get_stats(self):
        """Get current statistics"""
        with self.lock:
            if len(self.predictions_history) == 0:
                return {
                    'total': 0,
                    'toxic': 0,
                    'non_toxic': 0,
                    'toxicity_percentage': 0,
                    'avg_toxicity': 0,
                    'max_toxicity': 0,
                    'min_toxicity': 0,
                    'recent_count': 0
                }

            recent_scores = [p['toxicity_score'] for p in self.predictions_history]

            return {
                'total': self.total_predictions,
                'toxic': self.toxic_count,
                'non_toxic': self.total_predictions - self.toxic_count,
                'toxicity_percentage': (self.toxic_count / max(self.total_predictions, 1)) * 100,
                'avg_toxicity': round(np.mean(recent_scores), 4),
                'max_toxicity': round(max(recent_scores), 4),
                'min_toxicity': round(min(recent_scores), 4),
                'recent_count': len(self.predictions_history),
                'recent_predictions': list(self.predictions_history)[-10:]  # Last 10 predictions
            }

    def print_report(self):
        """Print monitoring report"""
        stats = self.get_stats()
        print("\n" + "="*60)
        print("📊 MONITORING DASHBOARD REPORT")
        print("="*60)
        print(f"Total Predictions: {stats['total']}")
        print(f"Toxic Comments Found: {stats['toxic']}")
        print(f"Safe Comments: {stats['non_toxic']}")
        print(f"Toxicity Rate: {stats['toxicity_percentage']:.2f}%")
        print(f"Average Toxicity Score: {stats['avg_toxicity']:.3f}")
        print(f"Max Toxicity Score: {stats['max_toxicity']:.3f}")
        print(f"Min Toxicity Score: {stats['min_toxicity']:.3f}")
        print("="*60)

    def reset(self):
        """Reset dashboard statistics"""
        with self.lock:
            self.predictions_history.clear()
            self.total_predictions = 0
            self.toxic_count = 0
        print("✅ Dashboard statistics reset!")

# Initialize dashboard
dashboard = MonitoringDashboard()
print("✅ Monitoring dashboard initialized!")

# Update the predict function to log to dashboard
# We need to modify the existing predict function to include dashboard logging
# Let's create a wrapper or update the existing route

# Note: The dashboard route will be added in the main Flask app
print("📊 Dashboard will automatically track all predictions")

✅ Monitoring dashboard initialized!
📊 Dashboard will automatically track all predictions


In [ ]:
# Add this dashboard panel to your HTML_TEMPLATE (add after the main card)
DASHBOARD_HTML = """
<!-- Dashboard Panel -->
<div class="main-card" style="margin-top: 20px;">
    <h2>📊 Live Monitoring Dashboard</h2>
    <div id="dashboardStats">
        <div style="text-align: center; padding: 20px;">
            <div class="spinner"></div>
            <p>Loading dashboard data...</p>
        </div>
    </div>
    <button onclick="refreshDashboard()" style="margin-top: 15px;">🔄 Refresh Dashboard</button>
    <button onclick="resetDashboard()" style="margin-top: 15px; background: #dc3545;">🗑️ Reset Statistics</button>
</div>

<script>
    function refreshDashboard() {
        fetch('/dashboard_stats')
            .then(response => response.json())
            .then(data => {
                if (data.success) {
                    updateDashboard(data.data);
                }
            })
            .catch(error => console.error('Error:', error));
    }

    function updateDashboard(stats) {
        const dashboardHtml = `
            <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-top: 20px;">
                <div class="info-card" style="text-align: center;">
                    <h3>Total Predictions</h3>
                    <div style="font-size: 2em; color: #2a5298;">${stats.total}</div>
                </div>
                <div class="info-card" style="text-align: center;">
                    <h3>Toxic Comments</h3>
                    <div style="font-size: 2em; color: #dc3545;">${stats.toxic}</div>
                </div>
                <div class="info-card" style="text-align: center;">
                    <h3>Safe Comments</h3>
                    <div style="font-size: 2em; color: #28a745;">${stats.non_toxic}</div>
                </div>
                <div class="info-card" style="text-align: center;">
                    <h3>Toxicity Rate</h3>
                    <div style="font-size: 2em; color: #ffc107;">${stats.toxicity_percentage.toFixed(1)}%</div>
                </div>
                <div class="info-card" style="text-align: center;">
                    <h3>Avg Toxicity</h3>
                    <div style="font-size: 1.5em;">${stats.avg_toxicity}</div>
                </div>
                <div class="info-card" style="text-align: center;">
                    <h3>Max Toxicity</h3>
                    <div style="font-size: 1.5em; color: #dc3545;">${stats.max_toxicity}</div>
                </div>
            </div>
            <div class="stats" style="margin-top: 20px;">
                <h3>Recent Predictions (Last 10)</h3>
                <div style="max-height: 300px; overflow-y: auto;">
                    ${stats.recent_predictions.map(pred => `
                        <div style="padding: 10px; border-bottom: 1px solid #e0e0e0; ${pred.is_toxic ? 'background: #fee;' : ''}">
                            <strong>${pred.is_toxic ? '⚠️ TOXIC' : '✅ SAFE'}</strong>
                            Score: ${(pred.toxicity_score * 100).toFixed(1)}%<br>
                            <small>${pred.text.substring(0, 80)}...</small><br>
                            <small style="color: #666;">${new Date(pred.timestamp).toLocaleString()}</small>
                        </div>
                    `).join('')}
                </div>
            </div>
        `;
        document.getElementById('dashboardStats').innerHTML = dashboardHtml;
    }

    function resetDashboard() {
        if (confirm('Are you sure you want to reset all dashboard statistics?')) {
            fetch('/dashboard_reset', { method: 'POST' })
                .then(response => response.json())
                .then(data => {
                    if (data.success) {
                        alert('Dashboard statistics reset successfully!');
                        refreshDashboard();
                    }
                })
                .catch(error => console.error('Error:', error));
        }
    }

    // Refresh dashboard every 5 seconds
    setInterval(refreshDashboard, 5000);
    // Initial load
    setTimeout(refreshDashboard, 1000);
</script>
"""

# Note: Integrate this into your HTML_TEMPLATE by adding DASHBOARD_HTML
# right before the closing </div> tag of the container

In [ ]:
def run_comprehensive_tests():
    """
    Run comprehensive tests on the detector
    """
    print("\n" + "="*70)
    print("🧪 COMPREHENSIVE TESTING SUITE")
    print("="*70)

    # Test categories
    test_cases = {
        "Clean/Positive": [
            "This is wonderful news!",
            "Thank you for your help.",
            "Great job everyone!",
            "I really appreciate this.",
            "You're doing amazing work."
        ],
        "Mildly Negative": [
            "This is disappointing.",
            "I'm not happy with this.",
            "This could be better.",
            "Not what I expected.",
            "This is frustrating."
        ],
        "Toxic/Hate Speech": [
            "You are stupid and worthless!",
            "I hate you so much!",
            "Go kill yourself!",
            "You're a complete idiot.",
            "Nobody wants you here."
        ],
        "Mixed Content": [
            "I hate this feature but love the concept.",
            "You're wrong, but I respect your opinion.",
            "This is terrible work, but you tried.",
            "I disagree strongly, however, valid points."
        ],
        "Edge Cases": [
            "",  # Empty string
            "   ",  # Whitespace only
            "A" * 1000,  # Very long text
            "!!!###$$$",  # Special characters only
            "normal comment with some !!! emphasis"
        ]
    }

    results_summary = {}

    for category, comments in test_cases.items():
        print(f"\n📂 Category: {category}")
        print("-"*50)

        category_results = []
        for comment in comments:
            if comment.strip():  # Skip empty
                result = detector.predict_single(comment)
                category_results.append(result)
                print(f"  {'⚠️' if result['is_toxic'] else '✅'} {comment[:50]:50} | Score: {result['toxicity_score']:.3f}")

        # Category statistics
        toxic_count = sum(1 for r in category_results if r['is_toxic'])
        avg_score = np.mean([r['toxicity_score'] for r in category_results]) if category_results else 0

        results_summary[category] = {
            'total': len(category_results),
            'toxic': toxic_count,
            'avg_score': avg_score
        }

        print(f"  📊 Summary: {toxic_count}/{len(category_results)} toxic, Avg Score: {avg_score:.3f}")

    # Print overall summary
    print("\n" + "="*70)
    print("📊 OVERALL TEST SUMMARY")
    print("="*70)

    for category, stats in results_summary.items():
        print(f"{category:20} | Total: {stats['total']:3} | Toxic: {stats['toxic']:3} | Avg Score: {stats['avg_score']:.3f}")

    return results_summary

# Run tests
test_results = run_comprehensive_tests()


🧪 COMPREHENSIVE TESTING SUITE

📂 Category: Clean/Positive
--------------------------------------------------
  ✅ This is wonderful news!                            | Score: 0.001
  ✅ Thank you for your help.                           | Score: 0.001
  ✅ Great job everyone!                                | Score: 0.001
  ✅ I really appreciate this.                          | Score: 0.001
  ✅ You're doing amazing work.                         | Score: 0.001
  📊 Summary: 0/5 toxic, Avg Score: 0.001

📂 Category: Mildly Negative
--------------------------------------------------
  ✅ This is disappointing.                             | Score: 0.001
  ✅ I'm not happy with this.                           | Score: 0.004
  ✅ This could be better.                              | Score: 0.001
  ✅ Not what I expected.                               | Score: 0.001
  ✅ This is frustrating.                               | Score: 0.001
  📊 Summary: 0/5 toxic, Avg Score: 0.002

📂 Category: Toxic/Hate Spee

In [ ]:
# Example: Process multiple comments from a file or list
def process_comments_file(comments_list, output_file="toxicity_report.json"):
    """
    Process a list of comments and save results
    """
    print("\n📊 Processing batch of comments...")

    # Analyze all comments
    stats = detector.analyze_batch_with_stats(comments_list)

    # Print summary
    print(f"\n✅ Batch Analysis Complete!")
    print(f"   Total Comments: {stats['total']}")
    print(f"   Toxic Comments: {stats['toxic_count']}")
    print(f"   Safe Comments: {stats['non_toxic_count']}")
    print(f"   Toxicity Rate: {stats['toxicity_percentage']:.2f}%")
    print(f"   Average Toxicity: {stats['avg_toxicity']:.3f}")

    # Save to JSON
    with open(output_file, 'w') as f:
        # Convert results to serializable format
        serializable_stats = {
            'total': stats['total'],
            'toxic_count': stats['toxic_count'],
            'non_toxic_count': stats['non_toxic_count'],
            'toxicity_percentage': stats['toxicity_percentage'],
            'avg_toxicity': stats['avg_toxicity'],
            'max_toxicity': stats['max_toxicity'],
            'min_toxicity': stats['min_toxicity'],
            'results': [
                {
                    'text': r['text'],
                    'toxicity_score': r['toxicity_score'],
                    'is_toxic': r['is_toxic'],
                    'label': r['label']
                }
                for r in stats['results']
            ]
        }
        json.d

In [ ]:
# Test the complete system with dashboard integration
print("\n" + "="*60)
print("🎯 TESTING COMPLETE SYSTEM WITH DASHBOARD")
print("="*60)

# Test comments
test_comments_batch = [
    "This is a great project!",
    "You are stupid and worthless!",
    "I appreciate your hard work.",
    "Go away, nobody likes you!",
    "Thanks for the help!",
    "You're an idiot.",
    "Wonderful discussion everyone!"
]

print("\n📝 Testing multiple comments...")
for comment in test_comments_batch:
    result = detector.predict_single(comment)
    dashboard.add_prediction(result)
    print(f"  {result['label']:10} | Score: {result['toxicity_score']:.3f} | {comment[:40]}")

# Show dashboard report
dashboard.print_report()

print("\n✅ System test complete!")
print("🌐 Access the web interface with live dashboard at:")
print(f"   {ngrok.get_tunnels()[0].public_url}")


🎯 TESTING COMPLETE SYSTEM WITH DASHBOARD

📝 Testing multiple comments...
  NON-TOXIC  | Score: 0.001 | This is a great project!
  TOXIC      | Score: 0.990 | You are stupid and worthless!
  NON-TOXIC  | Score: 0.001 | I appreciate your hard work.
  TOXIC      | Score: 0.717 | Go away, nobody likes you!
  NON-TOXIC  | Score: 0.001 | Thanks for the help!
  TOXIC      | Score: 0.986 | You're an idiot.
  NON-TOXIC  | Score: 0.001 | Wonderful discussion everyone!

📊 MONITORING DASHBOARD REPORT
Total Predictions: 7
Toxic Comments Found: 3
Safe Comments: 4
Toxicity Rate: 42.86%
Average Toxicity Score: 0.385
Max Toxicity Score: 0.990
Min Toxicity Score: 0.001

✅ System test complete!
🌐 Access the web interface with live dashboard at:
   https://cogitative-nonflawed-miyoko.ngrok-free.dev
